# Adsorption Analysis of Malachite Green
**Project:** Comprehensive study of Malachite Green dye removal using porous adsorbent.
**Author:** Mianjian, Chemical Engineering

## Objectives
This notebook automates the calculation of:
1.  **Kinetics:** Pseudo-First Order (PFO) & Pseudo-Second Order (PSO).
2.  **Isotherms:** Langmuir & Freundlich models.
3.  **Thermodynamics:** Calculation of $\Delta G^\circ$, $\Delta H^\circ$, and $\Delta S^\circ$ using Van't Hoff plot.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import linregress

%matplotlib inline

INPUT_FILE = 'malachite_data.xlsx'
MW_MALACHITE = 364.91
R_GAS = 8.314

xls = pd.ExcelFile(INPUT_FILE)
print("Excel file loaded successfully.")

In [ ]:
# --- Kinetics Models ---
def pfo_model(t, qe, k1):
    return qe * (1 - np.exp(-k1 * t))

def pso_model(t, qe, k2):
    return (k2 * (qe**2) * t) / (1 + k2 * qe * t)

# --- Isotherm Models ---
def langmuir_model(Ce, qm, KL):
    return (qm * KL * Ce) / (1 + KL * Ce)

def freundlich_model(Ce, KF, n):
    return KF * np.power(Ce, 1/n)

## 1. Adsorption Kinetics
In this section, we analyze the rate of adsorption to determine the mechanism (physical vs chemical).

**Models used:**
* **Pseudo-First Order (PFO):** $q_t = q_e (1 - e^{-k_1 t})$
* **Pseudo-Second Order (PSO):** $q_t = \frac{k_2 q_e^2 t}{1 + k_2 q_e t}$

In [ ]:
print("--- KINETICS ANALYSIS ---")
df_kin = pd.read_excel(xls, sheet_name='Kinetics')
t = df_kin['Time'].values
qt = df_kin['qt'].values

# Fit PFO
try:
    popt_pfo, _ = curve_fit(pfo_model, t, qt, p0=[max(qt), 0.01])
    qe_pfo, k1 = popt_pfo
    r2_pfo = np.corrcoef(qt, pfo_model(t, *popt_pfo))[0,1]**2
    print(f"PFO: qe={qe_pfo:.2f}, k1={k1:.4f}, R2={r2_pfo:.4f}")
except: popt_pfo = None

# Fit PSO
try:
    popt_pso, _ = curve_fit(pso_model, t, qt, p0=[max(qt), 0.001])
    qe_pso, k2 = popt_pso
    r2_pso = np.corrcoef(qt, pso_model(t, *popt_pso))[0,1]**2
    print(f"PSO: qe={qe_pso:.2f}, k2={k2:.4f}, R2={r2_pso:.4f}")
except: popt_pso = None

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(t, qt, color='black', label='Exp Data')
t_smooth = np.linspace(0, max(t)*1.1, 100)
if popt_pfo is not None: plt.plot(t_smooth, pfo_model(t_smooth, *popt_pfo), 'b--', label='PFO')
if popt_pso is not None: plt.plot(t_smooth, pso_model(t_smooth, *popt_pso), 'r-', label='PSO')
plt.xlabel('Time (min)'); plt.ylabel('qt (mg/g)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 2. Adsorption Isotherms
We fit equilibrium data to find the maximum adsorption capacity ($q_m$) and surface properties.

**Models used:**
* **Langmuir:** Assumes monolayer adsorption on a homogenous surface.
    $$q_e = \frac{q_m K_L C_e}{1 + K_L C_e}$$
* **Freundlich:** Assumes multilayer adsorption on a heterogeneous surface.
    $$q_e = K_F C_e^{1/n}$$

In [ ]:
print("--- ISOTHERM ANALYSIS ---")
df_iso = pd.read_excel(xls, sheet_name='Isotherm')
Ce = df_iso['Ce'].values
qe = df_iso['qe'].values

# Langmuir
popt_lan, _ = curve_fit(langmuir_model, Ce, qe, p0=[max(qe), 0.01])
print(f"Langmuir: qm={popt_lan[0]:.2f}, KL={popt_lan[1]:.4f}")

# Freundlich
popt_fre, _ = curve_fit(freundlich_model, Ce, qe, p0=[10, 2])
print(f"Freundlich: KF={popt_fre[0]:.2f}, n={popt_fre[1]:.2f}")

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(Ce, qe, color='black', label='Exp Data')
Ce_smooth = np.linspace(0, max(Ce)*1.1, 100)
plt.plot(Ce_smooth, langmuir_model(Ce_smooth, *popt_lan), 'b--', label='Langmuir')
plt.plot(Ce_smooth, freundlich_model(Ce_smooth, *popt_fre), 'r-', label='Freundlich')
plt.xlabel('Ce (mg/L)'); plt.ylabel('qe (mg/g)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 3. Thermodynamic Analysis
To determine if the process is spontaneous and exo/endothermic.

**Methodology:**
1.  Fit **Langmuir model** at each temperature (298, 308, 318, 328 K) to find $K_L$.
2.  Convert $K_L$ (L/mg) to dimensionless equilibrium constant $K_{eq}$ using Molecular Weight ($M_w = 364.91$ g/mol).
3.  Apply **Van't Hoff Equation**:
    $$\ln K_{eq} = -\frac{\Delta H^\circ}{R} \frac{1}{T} + \frac{\Delta S^\circ}{R}$$

In [ ]:
print("--- THERMODYNAMICS ---")
df_thermo = pd.read_excel(xls, sheet_name='Thermo')
temps = sorted(df_thermo['T'].unique())

kc_list, inv_T = [], []

for T in temps:
    sub = df_thermo[df_thermo['T'] == T]
    # finding K
    try:
        popt, _ = curve_fit(langmuir_model, sub['Ce'], sub['qe'], p0=[max(sub['qe']), 0.05])
        K_eq = popt[1] * 1000 * MW_MALACHITE
        kc_list.append(K_eq)
        inv_T.append(1/T)
    except: pass

# Van't Hoff
ln_K = np.log(kc_list)
slope, intercept, r_val, _, _ = linregress(inv_T, ln_K)
dH = -slope * R_GAS
dS = intercept * R_GAS

print(f"dH = {dH/1000:.2f} kJ/mol")
print(f"dS = {dS:.2f} J/mol.K")

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(inv_T, ln_K, s=80, color='purple')
plt.plot(inv_T, slope*np.array(inv_T) + intercept, 'k--')
plt.xlabel('1/T'); plt.ylabel('ln K'); plt.grid(True, alpha=0.3)
plt.title(f'Thermodynamics (dH={dH/1000:.1f} kJ/mol)')
plt.show()